# DriveSense-VLM — 01: SFT Training

**GPU**: A100 80GB (required) | **Time**: ~3–5 h | **Cost**: ~50 CU

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **RECOVERY**: If Colab disconnects mid-training, rerun Cells 2–3 (setup + install),
> then rerun the training cell. It auto-resumes from the latest Drive checkpoint.

**Prerequisites**: Run `00_data_pipeline.ipynb` first to generate `sft_train.jsonl`.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install training dependencies
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate datasets bitsandbytes wandb -q 2>&1 | tail -1
# Flash-attn is optional — takes 15+ min to compile; SDPA fallback works fine.
# Uncomment to install: !pip install flash-attn --no-build-isolation -q 2>&1 | tail -3

import torch
assert torch.cuda.is_available(), "No GPU! Set Runtime → A100"
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name} ({vram_gb:.0f} GB)")
if "A100" not in gpu_name:
    print(f"  WARNING: Expected A100 — got {gpu_name}. May OOM on <40 GB GPUs.")

import drivesense
print("✓ DriveSense loaded")

In [ ]:
# Verify SFT data exists and image paths are populated
import os, json

SFT_DIR = f"{OUTPUTS_ROOT}/data/sft_ready"
print("SFT Data Verification:")
print("─" * 48)
for split in ("train", "val", "test"):
    path = f"{SFT_DIR}/sft_{split}.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            count = sum(1 for _ in f)
        with open(path) as f:
            ex = json.loads(f.readline())
        img = ""
        for msg in ex.get("messages", []):
            for item in (msg.get("content") if isinstance(msg.get("content"), list) else []):
                if item.get("type") == "image":
                    img = item.get("image", "")
        img_ok = "✓" if img and os.path.exists(img) else "✗ empty/missing"
        print(f"  ✓ sft_{split}.jsonl: {count} examples | image: {img_ok}")
    else:
        raise FileNotFoundError(
            f"sft_{split}.jsonl not found at {path}\n"
            "Run 00_data_pipeline.ipynb first."
        )

In [ ]:
# Configure Weights & Biases
import wandb
wandb.login()  # Prompts for API key — get it at https://wandb.ai/authorize
print("✓ W&B configured — metrics will log to project: drivesense-vlm")

## Debug Training (10 steps)

Validates the training pipeline before committing to a full run.
Creates `configs/training_debug.yaml` in the repo directory (not in /tmp/ which breaks
multi-file config resolution).

In [ ]:
import os, yaml

os.chdir(REPO_ROOT)

# Create debug config alongside training.yaml so relative includes resolve correctly
with open("configs/training.yaml") as f:
    config = yaml.safe_load(f)

config["training"]["max_steps"]             = 10
config["training"]["num_epochs"]            = 1
config["training"]["logging_steps"]         = 1
config["training"]["save_strategy"]         = "no"
config["training"]["load_best_model_at_end"]= False
config["training"]["eval_strategy"]         = "no"

with open("configs/training_debug.yaml", "w") as f:
    yaml.dump(config, f)
print("✓ configs/training_debug.yaml written")

!python scripts/run_training.py --config configs/training_debug.yaml 2>&1

## Full Training

Checkpoints saved to Google Drive every 250 steps (persistent across disconnects).

**RECOVERY**: Rerun Cells 2–3, then rerun this cell. `--resume` auto-detects
the latest checkpoint on Drive.

In [ ]:
import os

os.chdir(REPO_ROOT)

SFT_DIR   = f"{OUTPUTS_ROOT}/data/sft_ready"
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

# Copy SFT data to fast ephemeral local SSD for faster DataLoader I/O
print("Copying SFT data to local /content/ for fast I/O...")
!cp {SFT_DIR}/sft_train.jsonl /content/sft_train.jsonl 2>/dev/null || true
!cp {SFT_DIR}/sft_val.jsonl   /content/sft_val.jsonl   2>/dev/null || true
print("✓ SFT data on fast local storage")

# Run training — checkpoints go to Drive (persistent)
!python scripts/run_training.py \
    --config configs/training.yaml \
    --override training.output_dir={TRAIN_OUT} \
    --override training.save_steps=250 \
    --override training.save_total_limit=3 \
    --resume \
    2>&1

## Results

In [ ]:
import os, json, glob

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

metrics_path = f"{TRAIN_OUT}/training_metrics.json"
if os.path.exists(metrics_path):
    m = json.loads(open(metrics_path).read())
    print("Training Metrics:")
    print("─" * 40)
    for k, v in m.items():
        print(f"  {k:<30}: {v}")
else:
    print(f"Metrics file not found at {metrics_path}")

best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

if best and os.path.exists(best):
    print(f"\n✓ Adapter saved: {best}")
    !ls -lh {best}
    print("\nProceed to 02_optimization.ipynb")
else:
    print("⚠ No checkpoint found — training may still be running or failed.")